<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Omit uncertainties for now. Add them later. Focus on getting the Neural net trained first
#1. Optimize Gradient Descent functions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
data_df_2020 = data_df_2020.drop(columns=[0,4]).reset_index(drop=True)
data_df_2020.columns = range(data_df_2020.columns.size)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8
0,1,1,2,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,2,1,3,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,1,2,3,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,0,3,3,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,3,1,4,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [ ]:
og_data = data_df_2020.copy()


In [6]:
input = np.array(data_df_2020.iloc[:,[0,1,2,3,7]].values) #Neutrons, Protons, Mass Number, Mass Excess, and Atomic Mass are take as inputs
output = np.array(data_df_2020.iloc[:,[5]].values) #Binding Energy is our output


In [7]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.2)

#Standardize input and output training data for simpler training
scaler_input = StandardScaler()
scaler_output = StandardScaler()
#must have two different scaler functions since input and output have different column dimensions

input_train = scaler_input.fit_transform(input_train)
output_train = scaler_output.fit_transform(output_train)

input_test = scaler_input.transform(input_test)
output_test = scaler_output.transform(output_test)

In [8]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [9]:
training_data = dataset(input_train,output_train)
testing_data = dataset(input_test,output_test)

In [26]:
train_dataloader = DataLoader(training_data,batch_size = 256,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 256,shuffle = True)

In [11]:
for i,b in train_dataloader:
  print(i)
  print("===========")
  print(b)
  break

tensor([[-1.4466, -1.6227, -1.5264,  0.2355,  0.7134],
        [-0.4375, -0.3345, -0.4002, -1.1602,  0.4917],
        [ 0.2276,  0.6674,  0.4023, -0.0179,  0.6731],
        [ 0.8469,  0.9537,  0.8950,  0.1581,  0.7011],
        [ 0.6863,  0.6316,  0.6698, -0.2907,  0.6298],
        [-1.5613, -1.9090, -1.7094,  1.5800, -1.6262],
        [ 0.0671,  0.4885,  0.2334, -0.3445,  0.6213],
        [-0.3458, -0.6924, -0.4846, -0.3422,  0.6216],
        [-1.1714, -1.0860, -1.1463, -0.7306,  0.5600],
        [-0.3687, -0.1556, -0.2875, -1.0414,  0.5106],
        [-1.1255, -0.8713, -1.0337, -0.5531,  0.5882],
        [-0.1164, -0.0125, -0.0764, -1.1109,  0.4996],
        [ 1.7643,  1.8125,  1.7960,  2.6138, -1.4621],
        [ 2.0395,  1.8482,  1.9791,  3.0537, -1.3922],
        [ 0.3423,  0.8463,  0.5431,  0.3487,  0.7314],
        [-1.1714, -0.7997, -1.0337, -0.0601,  0.6664]], device='cuda:0')
tensor([[ 0.1020],
        [ 0.8012],
        [-0.1313],
        [-0.1908],
        [ 0.0218],
       

In [12]:

class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],64)
    self.layer2 = nn.Linear(64,32)
    self.OutLayer = nn.Linear(32,1) #Output is Binding Energy
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.Relu(self.layer1(X))
    X = self.Relu(self.layer2(X))
    X = self.OutLayer(X)

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [ ]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             384
              ReLU-2                   [-1, 64]               0
            Linear-3                   [-1, 32]           2,080
              ReLU-4                   [-1, 32]               0
            Linear-5                    [-1, 1]              33
Total params: 2,497
Trainable params: 2,497
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [13]:
criterion = nn.MSELoss() #Loss function, uses Mean Squared Error
optimizer = Adam(BE_NeuralNet.parameters(), lr = 0.001) #Gradient Descent algorithm (Adam)

In [ ]:
for data in train_dataloader:
  inputs, output = data
  print(inputs)
  print(output)
  break

tensor([[-1.5879, -1.4285, -1.5370,  0.5982, -1.7955],
        [-0.7851, -1.1775, -0.9451,  0.2821,  0.7135],
        [ 0.4307,  0.7227,  0.5486, -0.1663,  0.6433],
        [ 1.8529,  1.6548,  1.7887,  2.3891, -1.5150],
        [ 0.1095,  0.1849,  0.1400, -0.9121,  0.5265],
        [-1.1062, -1.0700, -1.1001, -0.7710,  0.5486],
        [ 0.1095,  0.4358,  0.2386, -0.5373,  0.5852],
        [-0.9227, -0.8907, -0.9169, -0.8813,  0.5313]], device='cuda:0')
tensor([[-0.9633],
        [-0.0773],
        [-0.0557],
        [-0.8392],
        [ 0.4000],
        [ 1.1622],
        [ 0.1539],
        [ 1.0331]], device='cuda:0')


In [28]:
total_loss_train_plot = []
total_loss_test_plot = []
total_acc_train_plot = []
total_acc_test_plot = []

avg_loss = []

epochs = 20
for epoch in range(epochs):
  total_loss_train = 0
  total_acc_test = []
  total_loss_test = 0

  for data in train_dataloader:
    inputs, output = data

    prediction = BE_NeuralNet(inputs) #Prediction by neural network

    batch_loss = criterion(prediction,output) #calculate loss for current batch
    total_loss_train += batch_loss.item()

    batch_loss.backward() #Backpropagation done on each batch of data
    optimizer.step() #Adjust weights of parameters
    optimizer.zero_grad() #Reset gradient for next batch

  avg_loss_train = total_loss_train/len(train_dataloader) #Average training data loss (before backpropagation) over current epoch

  with torch.no_grad(): #Stop tracking gradients of test data (as it is not needed)
    for data in test_dataloader:
      inputs, output = data

      prediction = BE_NeuralNet(inputs)

      batch_loss = criterion(prediction,output)
      total_loss_test += batch_loss.item()

    avg_loss_test = total_loss_test/len(test_dataloader) #Average testing data loss (before backpropagation) over current epoch

    #Accuracy is not a strong metric for this particular neural network, which is predicting continuous values. Therefore, only the loss function is displayed.

    avg_loss.append(avg_loss_test)


  print(f'''Epoch: {epoch+1} Train_Loss: {avg_loss_train}
        Testing Loss: {avg_loss_test}''')
  print("=="*25)

avg_loss = sum(avg_loss)/len(avg_loss)
print(f"Avg Loss Final: {avg_loss}")

Epoch: 1 Train_Loss: 0.04519686145552745
        Testing Loss: 0.022235698997974396
Epoch: 2 Train_Loss: 0.04295551248408932
        Testing Loss: 0.021316701856752236
Epoch: 3 Train_Loss: 0.04309166876676803
        Testing Loss: 0.024989270294706028
Epoch: 4 Train_Loss: 0.04242728781294621
        Testing Loss: 0.02249594343205293
Epoch: 5 Train_Loss: 0.06924232246819884
        Testing Loss: 0.02543473554154237
Epoch: 6 Train_Loss: 0.05341477313777432
        Testing Loss: 0.025311154623826344
Epoch: 7 Train_Loss: 0.04406269104219973
        Testing Loss: 0.025839945146193106
Epoch: 8 Train_Loss: 0.04216751877296095
        Testing Loss: 0.02387555114304026
Epoch: 9 Train_Loss: 0.04136637016199529
        Testing Loss: 0.02423275417337815
Epoch: 10 Train_Loss: 0.041261561564169824
        Testing Loss: 0.022862952202558517
Epoch: 11 Train_Loss: 0.06684514346610133
        Testing Loss: 0.026521869624654453
Epoch: 12 Train_Loss: 0.05054420349188149
        Testing Loss: 0.02572364173